# HD–HR 模型的 population scaling：首轮修订方案

**状态：首轮代码依赖已实现；正式 multi-$N$ 实验尚未运行。**  
**修订日期：2026-07-23**

本版沿用 Vafidis.ipynb 的动力学符号，并采用不依赖任何参考规模的 population-mean 归一化。状态写作 $\vec I$、$\vec V$、$\vec r$，连接写作 $\mathbf W_{\mathrm{pre}\to\mathrm{post}}$，学习量写作 $\vec E_{\mathrm{HD}}$ 与 $\vec P_{\mathrm{pre}}$。

首轮目标是：**只在稠密通路中引入纯粹的 $1/N_{\mathrm{pre}}$，检验 intensive dynamics 是否随神经元数量收敛；其余模型改动延后。**


## 1. 首轮允许与禁止的改动

首轮只包含：

1. 记录现有各通路电流、rate、error、更新量和谱的 Phase 0 诊断；
2. 对稠密 HD→HD、LHR→HD、RHR→HD current 分别引入 $1/N_{\mathrm{pre}}$；
3. 建立不依赖特定神经元数量的 population-replication 测试；
4. 先做 frozen-weight 离散化控制，再做各 $N$ 在线学习；
5. 用 nested populations 和 common random numbers 做配对比较。

首轮不改变 sigmoid、bias、时间常数、visual cue、HD→HR 一对一通路、噪声、decoder，以及现有 predictive-learning 的 error、滤波、clipping 和积分顺序。

旧模型与新模型不再要求在某个特定 $N$ 上逐步等价；二者属于不同的尺度假设，历史 runs 只作为现象对照。


## 2. 纯 population-mean 参数化

对任一稠密 presynaptic population $X$，其 pathway current 定义为

$$\vec I_{X\to\mathrm{HD}}
=\frac{1}{N_X}\mathbf W_{X\to\mathrm{HD}}\vec r_X.$$

这里的 $\mathbf W$ 不再表示“直接求和时的有限网络有效权重”，而表示 continuum kernel 的离散采样。其元素、初始化尺度、learning rate 和 bounds 都是 intensive parameters：

$$W_{ij}=O(1),\qquad \eta=O(1),\qquad W_{\max}=O(1),$$

并在所有 $N$ 下使用同一组数值。forward current 中只出现由 presynaptic population size 决定的 $1/N_X$，不再包含参考规模或补偿 gain。

首次实现应从零或同一个与 $N$ 无关的小权重分布开始学习，并使用跨 $N$ 相同的 $\eta$ 与 $W_{\max}$。这些数值通过电流单位、error-decay timescale 和 clipping fraction 确定，而不是通过匹配某个特定 $N$ 的旧模型确定。

“自组织”并不意味着没有学习率或权重尺度，而是这些参数定义局部学习和突触强度，不随网络离散化规模改变。


## 3. 与 Vafidis 动力学一致的 scaled equation

沿用

$$\vec r_{\mathrm{HR}}=
\begin{bmatrix}\vec r_{\mathrm{LHR}}\\
\vec r_{\mathrm{RHR}}\end{bmatrix},\qquad
\mathbf W_{\mathrm{HR}\to\mathrm{HD}}=
\left[\mathbf W_{\mathrm{LHR}\to\mathrm{HD}},
\mathbf W_{\mathrm{RHR}\to\mathrm{HD}}\right].$$

HD distal input equation 改为

$$
\begin{aligned}
\tau_s\frac{\mathrm d\vec I_{\mathrm{HD},d}}{\mathrm dt}
={}&-\vec I_{\mathrm{HD},d}
+\frac{1}{N_{\mathrm{HD}}}
\mathbf W_{\mathrm{HD}\to\mathrm{HD}}\vec r_{\mathrm{HD}}\\
&+\frac{1}{N_{\mathrm{LHR}}}
\mathbf W_{\mathrm{LHR}\to\mathrm{HD}}\vec r_{\mathrm{LHR}}
+\frac{1}{N_{\mathrm{RHR}}}
\mathbf W_{\mathrm{RHR}\to\mathrm{HD}}\vec r_{\mathrm{RHR}}
-b_{\mathrm{HD}}\vec 1_{N_{\mathrm{HD}}}.
\end{aligned}
$$

LHR 与 RHR 是两个独立速度通路，因此分别除以自身的 population size。固定稀疏 $\mathbf W_{\mathrm{HD}\to\mathrm{HR}}$ 的 fan-in 不随 $N$ 增大，不做归一化；visual input 与逐细胞噪声也保持 $O(1)$。严格零均值随机连接的 $1/\sqrt N$ 不在首轮引入。


## 4. 保留 Vafidis predictive-learning 语义

继续使用

$$\vec E_{\mathrm{HD}}(t)
=f(\vec V_{\mathrm{HD},a})-f(\vec V_{\mathrm{HD},ss}),$$

$$\tau_P\frac{\mathrm d\vec P_{\mathrm{HD}}}{\mathrm dt}
=-\vec P_{\mathrm{HD}}+\vec r_{\mathrm{HD}},\qquad
\tau_P\frac{\mathrm d\vec P_{\mathrm{HR}}}{\mathrm dt}
=-\vec P_{\mathrm{HR}}+\vec r_{\mathrm{HR}}.$$

为避免把当前实现误写成 instantaneous outer product，权重更新统一记为

$$\frac{\mathrm d\mathbf W_{\mathrm{HD}\to\mathrm{HD}}}{\mathrm dt}
=\mathcal U_{\mathrm{HD}\to\mathrm{HD}}
(\vec E_{\mathrm{HD}},\vec P_{\mathrm{HD}},
\text{existing filter states};\eta_{\mathrm{HD}\to\mathrm{HD}}),$$

$$\frac{\mathrm d\mathbf W_{\mathrm{HR}\to\mathrm{HD}}}{\mathrm dt}
=\mathcal U_{\mathrm{HR}\to\mathrm{HD}}
(\vec E_{\mathrm{HD}},\vec P_{\mathrm{HR}},
\text{existing filter states};\eta_{\mathrm{HR}\to\mathrm{HD}}).$$

$\mathcal U$ 原样保留当前 PSP/prospective filtering、符号、clipping 与积分顺序。weight update 不额外乘或除以 $N$：当相关 presynaptic cells 占据固定角宽比例时，forward equation 中的 $N^{-1}\sum_j$ 使 pathway-level learning speed 保持 $O(1)$。


## 5. Population 数量的唯一语义

沿用 Vafidis.ipynb 的 population symbols：

- $N_{\mathrm{HD}}$：实际 HD cell 总数；
- $N_{\mathrm{HR}}$：实际 HR cell 总数；
- $N_{\mathrm{LHR}}=N_{\mathrm{HR}}/2$；
- $N_{\mathrm{RHR}}=N_{\mathrm{HR}}/2$。

当前 paired-HD geometry 的 unique preferred-direction 数另记为

$$N_{\mathrm{heading}}=N_{\mathrm{HD}}/2.$$

首轮固定 $N_{\mathrm{HR}}=N_{\mathrm{HD}}$，不同时扫描 HD/HR ratio。当前代码字段 **n_theta** 映射到 $N_{\mathrm{HD}}$，**n_hr** 映射到 $N_{\mathrm{HR}}$；内部派生 $N_{\mathrm{LHR}}$、$N_{\mathrm{RHR}}$、$N_{\mathrm{heading}}$ 并检查偶数约束。


## 6. Phase 0、1A 与 1B

### Phase 0：旧模型诊断

记录未经归一化的三条 pathway currents、$\vec r_{\mathrm{HD}}$、$\vec E_{\mathrm{HD}}$、weight-update norm、row sum 和工作点，用于确认原模型的尺度漂移；不把它用于校准新模型。

### Phase 1A：冻结权重的离散化控制

从同一个平滑 circular kernel $\mathcal W(\Delta\theta)$ 在不同网格采样：

$$W_{\mathrm{HD}\to\mathrm{HD},ij}^{(N)}
=\mathcal W_{\mathrm{HD}\to\mathrm{HD}}
(\theta_{\mathrm{HD},i}-\theta_{\mathrm{HD},j}).$$

用纯 $1/N$ dynamics 冻结权重，检查 pathway current、static bump、velocity gain、endpoint map、pinning 和完整 neural-state Jacobian。若 1A 不随 $N$ 收敛，说明 forward normalization 或角度离散化仍有错误。

### Phase 1B：现有规则从头在线学习

所有 $N$ 使用相同 intensive initialization、$\eta$、bounds、训练物理时长和完整 learning pipeline。每个 mouse seed 生成最大规模 master GP bank 后取 nested subsets；跨 $N$ 共用轨迹、shared angular noise 和数据切分。另设 independent-resampling condition 估计 population-realization 方差。


## 7. 验收标准

### 与 $N$ 无关的确定性测试

若把一个 presynaptic population 的每个单元、对应 firing rate 和权重列复制 $m$ 次，则

$$\frac{1}{mN_X}
\sum_{j=1}^{mN_X}W_{ij}r_j
=
\frac{1}{N_X}\sum_{j=1}^{N_X}W_{ij}r_j.$$

该 replication invariance 必须对 HD、LHR 和 RHR 三条稠密通路分别成立。一对一 HD→HR 的单细胞输入则在扩大 population 后保持不变，而不是除以 $N$。

### 跨 $N$ 收敛而非轨迹完全相同

成功标准是 normalized pathway current、population rate、bump width、learning timescale 和 velocity gain 趋于 $O(1)$ 平台，且 seed variance、sampling imbalance 与 pinning 总体减弱。RMSE 不要求严格单调下降，也允许有限 $N$ 最优点。

### 区分三类谱

- connectivity：$\mathbf W_{\mathrm{HD}\to\mathrm{HD}}/N_{\mathrm{HD}}$ 等 effective matrices；
- neural dynamics：冻结权重后的完整 neural-state block Jacobian；
- learning：$\lVert\vec E_{\mathrm{HD}}\rVert$ 与 $\lVert\mathrm d\mathbf W/\mathrm dt\rVert$。

旧 runs 的数值权重与新 intensive $\mathbf W$ 含义不同，不应直接混合作图或比较。


## 8. 首轮 pilot

建议使用

$$N_{\mathrm{HD}}=N_{\mathrm{HR}}\in\{60,120,240,480\},$$

对应 $N_{\mathrm{heading}}\in\{30,60,120,240\}$；所有规模地位相同，不设置特殊参考点。先运行 5–10 个 nested master seeds，再决定是否进行 30-mouse 大实验。

对 intensive observable 可探索

$$O(N_{\mathrm{HD}})=O_\infty+aN_{\mathrm{HD}}^{-\alpha},$$

但不预设所有指标都服从 $\alpha=1/2$。主要输出为三条 normalized pathway currents、rate/bump shape、cue anchoring、dark drift、velocity gain、endpoint map、full-state Jacobian 和 seed-level 分布。


## 9. 延后但保留的模型问题

以下内容不进入首个 scale patch：

- raw GP、只匹配 RMS 的 GP、删除 population common mode 的 GP 三种 visual conditions；
- Vafidis-faithful pair-shared 与 Clark-faithful pair-independent 两个并列模型族；
- softplus 与 population-mean inhibition；
- empirical-template overlap decoder；
- noise 无量纲化与严格中心化随机连接；
- local weight decay/minimum-norm 偏好。

这些项目只在纯 $1/N$ 版本显示初步尺度收敛后逐项加入。


## 10. 首轮实现边界

本轮已按以下顺序实现，入口为 `../configs/experiments/vafidis_population_mean_heterogeneous.yaml`：

1. Phase 0 pathway diagnostics；
2. 新的 population-mean normalization mode 与 run metadata；
3. HD→HD、LHR→HD、RHR→HD 三条通路分别除以 presynaptic $N$；
4. replication-invariance 和稀疏通路测试；
5. frozen-weight Phase 1A；
6. nested-seed Phase 1B pilot。

首个实现不提供参考规模参数，也不修改 visual generator、activation、noise、decoder 或 predictive-learning pipeline。新模型从统一的 intensive 参数化重新训练，而不是试图复现某个神经元数量下的旧模型。
